# 2교시 · 데이터 조회와 전처리
### — 필요한 것만 남기기

앞 시간에 데이터를 열었고, **결측치가 있다는 것**을 발견했습니다.
이번 시간에 그 결측치가 **왜** 생겼는지 알아냅니다.

**이 시간이 끝나면 할 수 있는 것**

1. 원하는 행과 열만 골라낼 수 있다
2. 조건에 맞는 데이터만 걸러낼 수 있다
3. 결측치·중복값·형식 오류를 처리할 수 있다
4. **처리 방법을 바꾸면 결론이 달라진다는 것을 안다**

---

> ### 이번 시간에 알아 둘 것
> 전처리는 데이터를 "깨끗하게 만드는" 작업이 아닙니다.
> **무엇을 버릴지 정하는 작업**입니다.
> 그리고 무엇을 버렸느냐에 따라 나중에 나오는 숫자가 달라집니다.

### 오늘 쓰는 데이터 — 문구·가구 유통사 주문 내역

| | |
|---|---|
| **무엇** | 어느 문구·가구 유통사의 주문 내역 (Tableau 공식 샘플 데이터) |
| **기간** | 2023-01-03 ~ 2026-12-30 (4년치) |
| **크기** | 10,239행 × 21열 · 주문 5,111건 · 고객 804명 |
| **한 줄은** | 주문이 아니라 **주문에 담긴 품목 하나**입니다 |
| **지역** | 미국(10,038) · 캐나다(201) |

**주요 열**

| 열 | 뜻 |
|---|---|
| `Order ID` · `Order Date` · `Ship Date` | 주문번호 · 주문일 · 배송일 |
| `Customer ID` · `Segment` | 고객 · 고객 유형(Consumer / Corporate / Home Office) |
| `Region` · `State/Province` · `City` | 지역(Central / East / South / West) · 주 · 도시 |
| `Category` · `Sub-Category` · `Product Name` | 대분류(3종) · 소분류(17종) · 제품명 |
| `Sales` · `Quantity` · `Discount` · `Profit` | 매출 · 수량 · 할인율 · 이익 |

> **결측치·이상치·중복값이 일부러 들어 있습니다.**
> 실무에서 받는 데이터가 그렇기 때문입니다. 손대지 않은 원본이 필요하면
> `superstore_orders_raw.csv` 를 쓰세요.

---

### 함께 쓰는 표 — 반품 목록

`superstore_returns.csv` · **296행 × 2열** (`Order ID` · `Returned`)

**반품된 주문번호만** 들어 있습니다. 반품되지 않은 주문은 이 표에 아예 없습니다.
그래서 주문 내역에 붙이면 반품 안 된 주문 쪽이 결측치가 됩니다.

In [ ]:
import pandas as pd

BASE = 'https://raw.githubusercontent.com/JasonWhiteLee/ak-data-analysis-basics/main/'

orders  = pd.read_csv(BASE + 'superstore_orders.csv', parse_dates=['Order Date', 'Ship Date'])
returns = pd.read_csv(BASE + 'superstore_returns.csv')

print(orders.shape)
orders.head(3)

---
# 2-1. 골라내기 — 열 고르기

## 열 하나 / 여러 개

앞 시간에 했던 것입니다. 대괄호 하나면 **Series**, 두 겹이면 **DataFrame**.

In [ ]:
orders['Sales'].head(3)

In [ ]:
orders[['Order Date', 'Category', 'Sales']].head(3)

## 필요 없는 열 버리기 — `.drop()`

`axis=1` 은 "열 방향"이라는 뜻입니다 (`axis=0` 은 행).

In [ ]:
# Row ID 는 그냥 번호라 분석에 쓸 일이 없습니다
small = orders.drop(['Row ID'], axis=1)

print('원래:', orders.shape)
print('버린뒤:', small.shape)

> ### 잠깐 — 이 열은 왜 남겨 둘까요
>
> 1교시에 `Country/Region` 을 봤습니다. 미국 10,038 / 캐나다 201 이었죠.
> 거의 다 미국인데, **완전히 하나는 아닙니다.**
>
> 이런 열은 버릴까요 남길까요? — **답은 "무엇을 하려는가"에 달려 있습니다.**
> 국내 매출만 볼 거면 캐나다를 빼야 하고, 전체를 볼 거면 남겨야 합니다.
>
> **버리기 전에 무엇을 버리는지 확인하는 것.** 이게 전처리의 전부입니다.

---
# 2-2. loc 와 iloc — 위치로 고르기

pandas에는 행·열을 고르는 전용 도구가 둘 있습니다. 이름이 비슷해서 늘 헷갈립니다.

| | 무엇으로 고르나 | 예 |
|---|---|---|
| **`.loc`** | **이름(라벨)** 으로 | `orders.loc[0, 'Sales']` |
| **`.iloc`** | **순서(번호)** 로 | `orders.iloc[0, 17]` |

`i` 는 **index(번호)** 의 i 라고 외우면 됩니다.

In [ ]:
# loc — [행, 열이름]
orders.loc[0, 'Sales']

In [ ]:
# iloc — [행번호, 열번호]
orders.iloc[0, 0]

## 범위로 고르기

`:` 는 "전부" 또는 "부터 까지"라는 뜻입니다.

In [ ]:
# 0~2행, Order Date 부터 Segment 까지
orders.loc[0:2, 'Order Date':'Segment']

In [ ]:
# 앞 3행, 앞 5열
orders.iloc[0:3, 0:5]

> ### 하나 다른 점
> `.loc[0:2]` 는 **2번 행을 포함**하고, `.iloc[0:2]` 는 **2번 행을 포함하지 않습니다.**
> 이름으로 고를 땐 끝을 포함하고, 번호로 자를 땐 포함하지 않습니다.
>
> 헷갈리면 **결과 행 수를 세어 보면 됩니다.** 오늘은 `.loc` 만 주로 씁니다.

---
# 2-3. 걸러내기 — 조건에 맞는 행만

**엑셀의 필터**와 같습니다. 오늘 가장 많이 쓸 기능입니다.

## 조건 하나

먼저 조건만 써 보면, `True` / `False` 가 줄줄이 나옵니다.

In [ ]:
(orders['Sales'] > 1000).head()

이 참·거짓 목록을 대괄호에 넣으면, **참인 행만** 남습니다.

In [ ]:
big_orders = orders[orders['Sales'] > 1000]

print(len(big_orders), '건')
big_orders[['Order Date', 'Category', 'Sales']].head()

## 조건 두 개 이상

- `&` = 그리고 (and)
- `|` = 또는 (or)
- **각 조건을 반드시 괄호로 감쌉니다.** 안 감싸면 에러가 납니다.

In [ ]:
cond = (orders['Category'] == 'Technology') & (orders['Sales'] > 1000)

tech_big = orders[cond]
print(len(tech_big), '건')
tech_big[['Category', 'Sales', 'Profit']].head()

## 자주 쓰는 조건 세 가지

In [ ]:
# 여러 값 중 하나 — isin
west_east = orders[orders['Region'].____]
print('isin      :', len(west_east))

# 사이 값 — between
mid_price = orders[orders['Sales'].between(100, 500)]
print('between   :', len(mid_price))

# 글자 포함 — str.contains
chairs = orders[orders['Product Name'].str.contains('Chair', case=False)]
print('contains  :', len(chairs))

## 날짜로 거르기

1교시에 `parse_dates` 로 날짜를 **진짜 날짜**로 읽었기 때문에 이런 게 됩니다.

In [ ]:
last_year = orders[orders['Order Date'] >= '2026-01-01']

print(len(last_year), '건')
print(last_year['Order Date'].min(), '~', last_year['Order Date'].max())

> **만약 `parse_dates` 를 안 했다면** 이 비교는 글자끼리 비교가 되어
> 엉뚱한 결과가 나오거나 에러가 납니다. 1교시 1-2의 "값의 종류"가 여기서 돌아옵니다.

## 정렬 — `.sort_values()`

In [ ]:
orders.sort_values('Sales', ascending=False)[['Product Name', 'Sales', 'Profit']].head(5)

> 가장 큰 주문의 `Sales` 를 보세요. 다음 시간에 다시 만납니다.

---
# 2-4. 결측치 다루기

비어 있는 값을 **결측치**라고 합니다. 다루는 법은 세 가지뿐입니다 —
**세고 · 버리고 · 채우기.** 하나씩 해 봅니다.

### ① 열별로 결측치가 몇 개인지 세세요

In [ ]:
orders.____.sum()

### ② `Ship Date` 의 결측치만 세세요

In [ ]:
print('Ship Date 결측치:', orders['Ship Date'].____.sum(), '개')

### ③ `Ship Date` 가 빈 행을 버리세요 — 몇 행이 사라집니까

In [ ]:
dropped = orders.____(subset=['Ship Date'])

print('버리기 전:', len(orders))
print('버린 후  :', len(dropped))

### ④ `Postal Code` 의 결측치를 `'미상'` 으로 채우세요

In [ ]:
orders['Postal Code'].____('미상').head(3)

## 셋 중 무엇을 고를까

| 방법 | 코드 | 언제 |
|---|---|---|
| **버린다** | `df.dropna(subset=['열'])` | 결측치가 무작위이고, 양이 적을 때 |
| **채운다** | `df['열'].fillna(값)` | 결측치의 의미를 알고, 대체값이 타당할 때 |
| **표시한다** | `df['열'].isna()` 로 새 열 | **결측치 자체가 정보일 때** |

> `fillna(0)` 을 습관적으로 쓰는 것이 가장 위험합니다.
> **비어 있는 것과 0은 다릅니다.** 매출이 "없는 것"과 "0원인 것"은 완전히 다른 이야기입니다.

## 그런데 — 버리기 전에 한 번 봐야 합니다

이 데이터의 `Ship Date` 결측치는 **아무데나 흩어져 있지 않습니다.**

| | 배송일이 빈 비율 |
|---|---|
| 반품된 주문 | **35.1%** |
| 반품 아닌 주문 | 0.4% |

반품되면 배송 기록이 제대로 안 남기 때문입니다.
**값이 없다는 사실 자체가 "반품됐다"를 알려 주고 있습니다.**

그래서 `dropna()` 로 지우면 반품 건이 유독 많이 사라지고,
**반품률이 5.79% 에서 5.19% 로 떨어집니다.** 아무도 거짓말하지 않았는데 숫자가 바뀝니다.

> `dropna()` 를 쓸지 말지는 **판단**입니다. 정해진 정답이 없습니다.
> 코드를 대신 써 주는 도구는 많지만, **이 판단은 대신 해 주지 않습니다.**

---
# 2-5. 중복 — 같은 줄이 두 번 들어왔을 때

시스템에서 두 번 내려받거나, 파일을 잘못 합치면 생깁니다.

In [ ]:
print('완전히 똑같은 행:', orders.duplicated().____)

In [ ]:
# 실제로 어떤 행인지 보기
orders[orders.duplicated(keep=False)].sort_values('Row ID').head(4)

In [ ]:
deduped = orders.drop_duplicates()

print('전:', len(orders))
print('후:', len(deduped))

> ### 주의 — 진짜 중복인지 확인하고 지웁니다
>
> 1교시에서 봤듯이 **한 주문에 여러 품목**이 있으면 `Order ID` 가 반복됩니다.
> 이건 중복이 아니라 정상입니다.
>
> `drop_duplicates()` 는 **모든 열이 동일한 행(완전 중복값)** 만 지웁니다.
> 특정 열 기준으로 지우려면 `subset=` 을 쓰는데, **이때 진짜 데이터가 날아갑니다.**
>
> ```python
> df.drop_duplicates(subset=['Order ID'])   # 주문당 1줄만 남김 = 품목 정보 소멸
> ```

---
# 2-6. 형식 바꾸기 — 계산이 안 되는 대부분의 원인

1교시 1-2에서 `'1500' + '900'` 이 `'1500900'` 이 되는 걸 봤습니다.
실제 데이터에서 이 문제가 어떻게 나타나는지 봅니다.

In [ ]:
orders.dtypes

## 숫자로 바꾸기 — `.astype()` / `pd.to_numeric()`

In [ ]:
# 예시 — 숫자처럼 생겼지만 글자인 열
sample = pd.DataFrame({'amount': ['1500', '900', '700']})
print(sample.dtypes)
print('더하면:', sample['amount'].sum())      # 이어붙습니다

In [ ]:
sample['amount'] = sample['amount'].astype(int)

print(sample.dtypes)
print('더하면:', sample['amount'].sum())      # 이제 계산됩니다

> **`errors='coerce'` 를 기억하세요.**
> 숫자로 못 바꾸는 값(예: `'미상'`)이 섞여 있으면 `astype` 은 에러를 냅니다.
> `pd.to_numeric(열, errors='coerce')` 를 쓰면 **못 바꾸는 값을 결측치로** 만들고 넘어갑니다.

## 날짜에서 조각 꺼내기 — `.dt`

날짜 열에 `.dt` 를 붙이면 연·월·요일을 꺼낼 수 있습니다.
**월별 집계를 하려면 반드시 필요합니다.** (다음 시간에 씁니다)

In [ ]:
orders['year']  = orders['Order Date'].dt.____
orders['month']    = orders['Order Date'].dt.month
orders['weekday']  = orders['Order Date'].dt.day_name()

orders[['Order Date', 'year', 'month', 'weekday']].head()

## 파생 열 만들기 — 없는 정보를 계산해 낸다

두 날짜의 차이로 **배송에 걸린 날짜**를 구할 수 있습니다.

In [ ]:
orders['ship_days'] = (orders['Ship Date'] - orders['Order Date']).dt.days

orders[['Order Date', 'Ship Date', 'ship_days']].head()

In [ ]:
orders['ship_days'].describe()

> `count` 를 보세요. 전체 행 수보다 적습니다.
> **`Ship Date` 가 빈 행은 배송일수도 자동으로 비어 있습니다.**
> 결측치는 이렇게 조용히 아래로 퍼져 나갑니다.

---
# 2-7. 이제 진짜 지저분한 데이터

지금까지 쓴 데이터는 정리된 편입니다.
**실무에서 받는 데이터는 이렇지 않습니다.**

영국의 어느 온라인 소매점 **1년치 거래 기록 54만 건**을 열어 봅니다.

### 새 데이터 — 영국 온라인 소매점 거래 기록

| | |
|---|---|
| **무엇** | 영국의 어느 온라인 소매점 거래 기록 (UCI 공개 데이터) |
| **기간** | 2010-12-01 ~ 2011-12-09 (1년치, 시각까지 기록) |
| **크기** | 541,909행 × 8열 |
| **한 줄은** | 송장에 찍힌 품목 한 줄 |

**열**

| 열 | 뜻 |
|---|---|
| `InvoiceNo` | 송장번호 — **`C` 로 시작하면 취소 건입니다** |
| `StockCode` · `Description` | 상품코드 · 상품명 |
| `Quantity` · `UnitPrice` | 수량 · 단가 |
| `InvoiceDate` | 거래 일시 |
| `CustomerID` | 고객번호 — **비회원 주문은 비어 있습니다** |
| `Country` | 국가 (38개국이지만 영국이 91%) |

> 앞의 주문 데이터와 달리 **손대지 않은 진짜 원본**입니다.
> 결측치·취소 건·이상한 단가가 원래부터 들어 있습니다.

In [ ]:
retail = pd.read_csv(BASE + 'online_retail.csv', parse_dates=['InvoiceDate'])

print(retail.shape)
retail.head()

In [ ]:
retail.info()

## 문제 1 — 고객 번호의 25%가 없습니다

In [ ]:
print('CustomerID 결측: {:,}건 ({:.1f}%)'.format(
      retail['CustomerID'].isna().sum(), retail['CustomerID'].isna().mean() * 100))

54만 건 중 13만 건에 고객 번호가 없습니다.

**이걸 지우면 매출의 4분의 1이 사라집니다.**
하지만 남기면 "고객당 구매액" 같은 분석을 할 수 없습니다.

> **정답이 없습니다.** 무엇을 하려는지에 따라 다릅니다.
> - 고객 분석 -> 지워야 함. 대신 **"비회원 제외"라고 보고서에 적어야 합니다**
> - 매출 집계 -> 남겨야 함
>
> 중요한 건 **어느 쪽을 골랐는지 밝히는 것**입니다.

## 문제 2 — 취소 주문이 섞여 있습니다

이 데이터는 취소 건의 송장번호가 **`C` 로 시작**합니다.

In [ ]:
retail['canceled'] = retail['InvoiceNo'].astype(str).str.startswith('C')

print('취소 건수:', retail['canceled'].sum())
retail[retail['canceled']].head(3)

`Quantity` 가 **음수**인 것을 보세요. 취소는 수량을 빼는 방식으로 기록돼 있습니다.

**모르고 매출을 합치면 어떻게 될까요?**

In [ ]:
retail['amount'] = retail['Quantity'] * retail['UnitPrice']

print('그대로 합계    : {:>12,.0f}'.format(retail['amount'].sum()))
print('취소 제외 합계 : {:>12,.0f}'.format(retail[~retail['canceled']]['amount'].sum()))

**90만 가까이 차이납니다.**

재미있는 건 **어느 쪽도 틀리지 않았다**는 점입니다.
- "실제로 들어온 돈"을 알고 싶으면 -> 취소를 포함한 순매출
- "얼마나 팔렸는지"를 알고 싶으면 -> 취소 제외한 총매출

> **숫자를 내기 전에 무엇을 알고 싶은지 먼저 정해야 합니다.**
> 순서가 반대가 되면, 이미 나온 숫자에 맞춰 질문을 끼워 맞추게 됩니다.

## 문제 3 — 단가가 0이거나 음수입니다

In [ ]:
print('단가 0 이하:', (retail['UnitPrice'] <= 0).sum(), '건')

retail[retail['UnitPrice'] <= 0]['Description'].value_counts().head(5)

설명을 보면 정체가 드러납니다 — 사은품, 재고 조정, 파손 처리 같은 것들입니다.

**이건 오류가 아니라 다른 종류의 기록입니다.**
매출 분석에서는 빼야 하지만, "왜 이렇게 많이 파손됐나"를 볼 거라면 이게 본체입니다.

## 문제 4 — 중복값이 5천 건 (모든 열이 동일한 행)

In [ ]:
print('완전중복:', retail.duplicated().sum(), '건')

## 문제 5 — 이건 어느 나라 데이터인가

In [ ]:
retail['Country'].value_counts().head(5)

38개국이 있지만 **영국이 91%** 입니다.

> "글로벌 온라인 소매 데이터"라고 부르면 틀린 말이 됩니다.
> **1교시의 질문이 여기서 다시 나옵니다 — 이 데이터는 누구를 대표합니까?**

---
# 2-8. 실습 — 배운 것으로 직접 정리해 보기

`retail` 을 네 단계로 정리합니다. **한 셀씩 빈칸을 채우고 실행**하세요.
전부 이번 시간에 나온 것들입니다.

### ① 중복값을 제거하세요

In [ ]:
step1 = retail.____

print('중복 제거 후 : {:>7,} 행'.format(len(step1)))

### ② 취소 건을 제외하세요 — `canceled` 가 False 인 것만 남깁니다

In [ ]:
step2 = step1[~step1[____]]

print('취소 제외 후 : {:>7,} 행'.format(len(step2)))

### ③ 단가(`UnitPrice`)가 0보다 큰 행만 남기세요

In [ ]:
step3 = step2[step2['UnitPrice'] ____ 0]

print('단가 0 제외  : {:>7,} 행'.format(len(step3)))

### ④ 고객번호(`CustomerID`)에 결측치가 있는 행을 제거하세요

In [ ]:
step4 = step3.____(subset=['CustomerID'])

print('비회원 제외  : {:>7,} 행'.format(len(step4)))

### 그래서 얼마나 버렸나

In [ ]:
print('시작 : {:>7,} 행'.format(len(retail)))
print('남음 : {:>7,} 행'.format(len(step4)))
print()
print('전체의 {:.1f}% 를 버렸습니다'.format((1 - len(step4) / len(retail)) * 100))

---
# 정리 — 오늘 쓴 것

## 코드

| 하는 일 | 코드 |
|---|---|
| 열 고르기 | `df['열']` · `df[['열1','열2']]` |
| 열 버리기 | `df.drop(['열'], axis=1)` |
| 이름으로 고르기 | `df.loc[행, '열']` |
| 번호로 고르기 | `df.iloc[행번호, 열번호]` |
| 조건 필터 | `df[df['열'] > 100]` |
| 조건 여러 개 | `df[(cond1) & (cond2)]` |
| 여러 값 중 하나 | `df['열'].isin([...])` |
| 사이 값 | `df['열'].between(a, b)` |
| 글자 포함 | `df['열'].str.contains('...')` |
| 정렬 | `df.sort_values('열', ascending=False)` |
| 결측치 세기 | `df.isna().sum()` · `df.isnull().sum()` |
| 결측치 행 버리기 | `df.dropna(subset=['열'])` |
| 결측치 채우기 | `df['열'].fillna(값)` |
| 중복 확인 / 제거 | `df.duplicated().sum()` · `df.drop_duplicates()` |
| 형 바꾸기 | `df['열'].astype(int)` · `pd.to_numeric(..., errors='coerce')` |
| 날짜 조각 | `df['날짜'].dt.year` · `.dt.month` · `.dt.day_name()` |
| 교차표 | `pd.crosstab(A, B)` |

## 남길 것 세 가지

1. **결측치를 지우기 전에 "왜 생겼는지" 묻는다** — 반품 건에 몰려 있었습니다
2. **버린 것은 결과를 바꾼다** — `dropna()` 한 줄로 반품률이 10% 줄었습니다
3. **무엇을 버렸는지 밝힌다** — 밝히지 않은 보고서는 검증할 수 없습니다

---

### 다음 시간

정리한 데이터를 **묶어서 요약하고, 다른 표와 이어 붙입니다.**
그리고 이런 질문을 하게 됩니다 — **"매출 3억"은 좋은 겁니까, 나쁜 겁니까?**

---
# 자주 쓰는 패턴

전처리는 **모양이 몇 개 안 됩니다.** 조건과 열 이름만 바뀝니다.

| 하고 싶은 것 | 패턴 |
|---|---|
| 조건으로 거르기 | `df[df['열'] > 값]` |
| 조건 두 개 | `df[(조건1) & (조건2)]` — 각 조건을 괄호로 |
| 여러 값 중 하나 | `df[df['열'].isin([...])]` |
| 사이 값 | `df[df['열'].between(a, b)]` |
| 글자 포함 | `df[df['열'].str.contains('...')]` |
| 정렬 | `df.sort_values('열', ascending=False)` |
| 결측치 세기 · 버리기 · 채우기 | `df.isna().sum()` · `df.dropna(subset=['열'])` · `df['열'].fillna(값)` |
| 중복 세기 · 지우기 | `df.duplicated().sum()` · `df.drop_duplicates()` |
| 형 바꾸기 | `df['열'].astype(int)` |
| 날짜 조각 | `df['날짜'].dt.year` · `.dt.month` · `.dt.day_name()` |

아래에서 **조건 필터 패턴을 열만 바꿔** 여러 번 써 봅니다.

### 패턴 `df[df['열'] > 값]` — ① 매출 1000 초과

In [ ]:
len(orders[orders['Sales'] ____ 1000])

### 같은 패턴 — ② 이익 0 미만 (적자)

In [ ]:
len(orders[orders[____] < 0])

### 같은 패턴 — ③ 할인율 0.5 이상

In [ ]:
len(orders[____])

### 패턴 `df[df['열'].isin([...])]` — ① 지역 두 곳

In [ ]:
len(orders[orders['Region'].____(['East', 'West'])])

### 같은 패턴 — ② 대분류 두 개

In [ ]:
len(orders[orders['Category'].____])

---
# 연습문제

지금까지 배운 것으로 직접 풀어 봅니다. **빈칸(`____`)을 채우고 실행**하세요.
막히면 위로 올라가 같은 함수를 쓴 곳을 찾아보거나, 맨 끝 정리표를 보세요.

> 답이 하나만 있는 건 아닙니다. **실행해서 원하는 결과가 나오면 맞은 것입니다.**

### 문제 1. `Category` 와 `Sales` 두 열만 뽑아 위 3줄을 보세요.

In [ ]:
orders[____].head(3)

### 문제 2. 이익(`Profit`)이 음수인 행만 남기고, 몇 건인지 세세요.

In [ ]:
loss = orders[orders['Profit'] ____]
print(len(loss), '건')

### 문제 3. `Region` 이 South 이면서 `Sales` 가 500 이상인 행을 고르세요.

In [ ]:
cond = (orders['Region'] == 'South') ____ (orders['Sales'] >= 500)
print(len(orders[cond]), '건')

### 문제 4. `Ship Mode` 가 First Class 또는 Same Day 인 행을 고르세요.

In [ ]:
fast = orders[orders['Ship Mode'].____]
print(len(fast), '건')

### 문제 5. 이익이 가장 큰 5건을 이익 내림차순으로 보세요.

In [ ]:
orders.____[['Sub-Category', 'Sales', 'Profit']].head(5)

### 문제 6. 완전히 똑같은 행이 몇 개인지 세세요.

In [ ]:
print('중복값:', orders.____.sum())

### 문제 7. 중복값을 지운 뒤 행 수를 출력하세요.

In [ ]:
print('제거 후:', len(orders.____))

### 문제 8. `Postal Code` 의 결측치를 문자열 `'없음'` 으로 채우세요.

In [ ]:
orders['Postal Code'].____.isna().sum()

### 문제 9. 주문일에서 **연도**만 꺼내 새 열 `order_year` 를 만드세요.

In [ ]:
orders['order_year'] = orders['Order Date'].dt.____
orders['order_year'].value_counts().sort_index()